# 수정 분기 분석 전체 실행

기존 고정 입력으로 **12 데이터셋 → 13 분기 매출액 분석 → 표본 보조진단 → 보고서 08**을 새 커널에서 순서대로 실행한다. 네트워크 요청 없이 실행하며 기존 기준선·시간창 분석 결과는 보존한다.

- 범위: 종로구·마포구, 외식업 10종, 2023~2025년 분기.
- 주노출: 상권 중심점에서 500m 이내 시설의 **분기 예정 시작회차 수**. 실제 회차·관객 수가 아니다.
- 주표본: 파싱 전 후보 ID 중 시작일정을 재구성할 수 있는 ID 비율 100%. 일정의 실제 정확도 100%를 뜻하지 않는다.
- 주결과: 분기 총매출액. 보조결과: 21~24시 매출액. 분기 고정 시간대 자료를 공연 직후 매출로 바꾸지 않는다.
- 결과를 보기 전에 고정한 계획: `metadata/revised_quarterly_plan.json`.

최초 원자료 준비는 기존 `00_전체실행.ipynb`가 담당한다. 현재 PC에는 해당 고정 자료가 준비돼 있다. 실행 중 다른 수집·분석 노트북을 동시에 실행하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, time
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter
from IPython.display import display
import pandas as pd
ROOT=PROJECT_ROOT
OUT=project_path('reports/20261006/revised_quarterly')
HTML=OUT/'notebooks'
HTML.mkdir(parents=True,exist_ok=True)
for var in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:os.environ[var]='1'
NAMES=['jupyter-notebook/20261006/notebooks/12_분기예정회차_데이터셋.ipynb','jupyter-notebook/20261006/notebooks/13_분기매출액_수정분석.ipynb','jupyter-notebook/20261006/report-notebooks/09_수정표본_보조진단.ipynb','jupyter-notebook/20261006/report-notebooks/08_수정분석_결과보고서.ipynb']
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
plan_path=project_path('jupyter-notebook/20261006/metadata/revised_quarterly_plan.json')
plan=project_json_loads(plan_path.read_text(encoding='utf-8'))
for name,expected in plan['protected_source_sha256'].items():assert sha(project_path(name))==expected,name
manifest={'started_utc':datetime.now(timezone.utc).isoformat(),'status':'running','steps':[],
          'network_requests':0,'plan_sha256':sha(plan_path)}
def save_manifest():
    (OUT/'execution_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
save_manifest()

## 실행과 결과 저장

실패하면 해당 단계의 실행 노트북과 실패 기록을 남기고 중단한다. 원본 파일을 수정하지 않는다. 결과 보고서 HTML은 마지막 노트북의 canonical artifact builder로 만든다.

In [ ]:
for name in NAMES:
    path=project_path(name)
    notebook=nbformat.read(path,as_version=4)
    source=chr(10).join(c.source for c in notebook.cells)
    started=time.perf_counter()
    print('실행:',path.name,flush=True)
    try:
        NotebookClient(notebook,timeout=900,kernel_name='python3',resources={'metadata':{'path':str(path.parent)}}).execute()
        nbformat.validate(notebook);nbformat.write(notebook,path)
        exporter=HTMLExporter(template_name='lab',exclude_input=True,exclude_input_prompt=True,exclude_output_prompt=True)
        body,_=exporter.from_notebook_node(notebook)
        html_path=HTML/(path.stem+'.html');html_path.write_text(body,encoding='utf-8')
        manifest['steps'].append({'notebook':name,'status':'passed','seconds':round(time.perf_counter()-started,2),
          'code_cells':sum(c.cell_type=='code' for c in notebook.cells),
          'source_sha256':hashlib.sha256(source.encode('utf-8')).hexdigest(),'executed_sha256':sha(path),
          'html':html_path.relative_to(ROOT).as_posix()})
        save_manifest()
    except Exception as exc:
        failed=OUT/'failed';failed.mkdir(exist_ok=True);nbformat.write(notebook,failed/path.name)
        manifest['status']='failed';manifest['failed_notebook']=name;manifest['error_type']=type(exc).__name__
        save_manifest();raise
for name,expected in plan['protected_source_sha256'].items():assert sha(project_path(name))==expected,name
assert sha(plan_path)==manifest['plan_sha256']
manifest['status']='completed';manifest['finished_utc']=datetime.now(timezone.utc).isoformat()
manifest['protected_baselines_unchanged']=True
save_manifest()
display(pd.DataFrame(manifest['steps'])[['notebook','status','seconds']])

## 결과 읽기

- 최종 보고서: `reports/20261006/revised_quarterly/report.html`
- 예정회차 데이터: `dataset/analysis/final/revised_quarterly/`
- 데이터 검증: `reports/20261006/revised_quarterly/12_*`
- 수정 회귀 결과·검증: `reports/20261006/revised_quarterly/13_*`
- 실행 기록: `reports/20261006/revised_quarterly/execution_manifest.json`

현재 자료로 확정 가능한 것은 예정 공연량과 분기 매출액의 관련성이다. 실제 관객 소비나 공연 종료 직후의 인과효과는 별도 자료가 필요하다.